In [ ]:
# Example 1: A model with pyro.sample and pyro.plate, inspected with poutine.trace
import torch
import pyro
import pyro.distributions as dist
from pyro import poutine
from pyro.poutine.util import site_is_subsample

print(f"pyro-ppl {pyro.__version__}, torch {torch.__version__}")
pyro.set_rng_seed(0)

def model(data):
    # Global latent variables
    mu = pyro.sample("mu", dist.Normal(0.0, 10.0))
    p = pyro.sample("p", dist.Beta(2.0, 2.0))
    # One conditionally independent observation per data point
    with pyro.plate("data", len(data)):
        pyro.sample("obs", dist.Normal(mu, 1.0), obs=data)
    return mu, p

data = torch.tensor([0.9, 1.1, 1.3, 0.7, 1.0])
trace = poutine.trace(model).get_trace(data)
trace.compute_log_prob()

sites = {name: site for name, site in trace.nodes.items() if site["type"] == "sample"}
print("sample sites:", list(sites))
# pyro.plate records its subsample indices as a site of its own, named after the plate
assert list(sites) == ["mu", "p", "data", "obs"], list(sites)
assert site_is_subsample(sites["data"]) and not site_is_subsample(sites["obs"])
assert [name for name, site in sites.items() if site["is_observed"]] == ["obs"]
assert trace.nodes["mu"]["value"].shape == torch.Size([])
assert 0.0 < float(trace.nodes["p"]["value"]) < 1.0
assert torch.equal(trace.nodes["obs"]["value"], data)
assert trace.nodes["obs"]["log_prob"].shape == data.shape, trace.nodes["obs"]["log_prob"].shape
data_plate = trace.nodes["obs"]["cond_indep_stack"][0]
assert (data_plate.name, data_plate.size) == ("data", len(data)), data_plate

# poutine.condition fixes a latent site to a value; the observed log-density then matches a hand computation
conditioned = poutine.trace(poutine.condition(model, data={"mu": torch.tensor(1.0)})).get_trace(data)
conditioned.compute_log_prob()
expected = dist.Normal(1.0, 1.0).log_prob(data).sum()
print(f"log p(obs | mu=1): {conditioned.nodes['obs']['log_prob_sum'].item():.4f} (by hand {expected.item():.4f})")
assert conditioned.nodes["mu"]["value"].item() == 1.0
assert torch.allclose(conditioned.nodes["obs"]["log_prob_sum"], expected)

In [ ]:
# Example 2: Stochastic variational inference on a Beta-Bernoulli model
import torch
import pyro
import pyro.distributions as dist
from pyro.distributions import constraints
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam

pyro.clear_param_store()
pyro.set_rng_seed(1)

# 30 coin flips with 22 heads; prior Beta(10, 10) gives the posterior Beta(32, 18)
flips = torch.cat([torch.ones(22), torch.zeros(8)])
prior_a, prior_b = 10.0, 10.0
post_a, post_b = prior_a + flips.sum().item(), prior_b + (len(flips) - flips.sum()).item()
analytic_mean = post_a / (post_a + post_b)

def coin_model(flips):
    f = pyro.sample("fairness", dist.Beta(prior_a, prior_b))
    with pyro.plate("flips", len(flips)):
        pyro.sample("obs", dist.Bernoulli(f), obs=flips)

def coin_guide(flips):
    a = pyro.param("alpha_q", torch.tensor(2.0), constraint=constraints.positive)
    b = pyro.param("beta_q", torch.tensor(2.0), constraint=constraints.positive)
    pyro.sample("fairness", dist.Beta(a, b))

elbo = Trace_ELBO(num_particles=4, vectorize_particles=True, max_plate_nesting=1)
svi = SVI(coin_model, coin_guide, Adam({"lr": 0.05}), loss=elbo)
losses = [svi.step(flips) for _ in range(500)]

alpha_q, beta_q = pyro.param("alpha_q").item(), pyro.param("beta_q").item()
fitted_mean = alpha_q / (alpha_q + beta_q)
first, last = sum(losses[:50]) / 50, sum(losses[-50:]) / 50
print(f"ELBO loss: first 50 steps {first:.2f}, last 50 steps {last:.2f}")
print(f"posterior mean: fitted {fitted_mean:.3f}, analytic {analytic_mean:.3f}")
assert last < first, (first, last)
assert abs(fitted_mean - analytic_mean) < 0.05, (fitted_mean, analytic_mean)

In [ ]:
# Example 3: MCMC with the No-U-Turn sampler on a one-dimensional Normal model
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS

pyro.set_rng_seed(2)
true_loc = 3.0
observations = true_loc + 0.5 * torch.randn(40)

def normal_model(y):
    loc = pyro.sample("loc", dist.Normal(0.0, 10.0))
    with pyro.plate("y", len(y)):
        pyro.sample("obs", dist.Normal(loc, 0.5), obs=y)

mcmc = MCMC(NUTS(normal_model), num_samples=50, warmup_steps=50, disable_progbar=True)
mcmc.run(observations)
samples = mcmc.get_samples()

print("sampled sites:", sorted(samples))
print(f"posterior mean of loc: {samples['loc'].mean():.3f} (data mean {observations.mean():.3f})")
assert sorted(samples) == ["loc"], sorted(samples)
assert samples["loc"].shape == torch.Size([50]), samples["loc"].shape
assert torch.isfinite(samples["loc"]).all()
assert abs(samples["loc"].mean().item() - observations.mean().item()) < 0.5

In [ ]:
# Example 4: A Bayesian linear layer with PyroModule and PyroSample
import torch
import pyro
import pyro.distributions as dist
from pyro import poutine
from pyro.nn import PyroModule, PyroSample
from pyro.poutine.util import site_is_subsample

pyro.set_rng_seed(3)

class BayesianRegression(PyroModule):
    def __init__(self, in_features):
        super().__init__()
        # PyroModule[torch.nn.Linear] is a torch.nn.Linear whose parameters can be random variables
        self.linear = PyroModule[torch.nn.Linear](in_features, 1)
        self.linear.weight = PyroSample(dist.Normal(0.0, 1.0).expand([1, in_features]).to_event(2))
        self.linear.bias = PyroSample(dist.Normal(0.0, 5.0).expand([1]).to_event(1))

    def forward(self, x, y=None):
        mean = self.linear(x).squeeze(-1)
        sigma = pyro.sample("sigma", dist.Uniform(0.0, 2.0))
        with pyro.plate("data", x.shape[0]):
            pyro.sample("obs", dist.Normal(mean, sigma), obs=y)
        return mean

regression = BayesianRegression(in_features=3)
assert isinstance(regression.linear, torch.nn.Linear)
x = torch.randn(8, 3)
trace = poutine.trace(regression).get_trace(x)
sites = {
    name: site["value"].shape
    for name, site in trace.nodes.items()
    if site["type"] == "sample" and not site_is_subsample(site)
}
print("sample sites:", sites)
assert sites["linear.weight"] == torch.Size([1, 3]), sites
assert sites["linear.bias"] == torch.Size([1]), sites
assert sites["obs"] == torch.Size([8]), sites
assert sorted(sites) == ["linear.bias", "linear.weight", "obs", "sigma"], sorted(sites)

# pyro.module registers the parameters of an ordinary torch.nn.Module in Pyro's parameter store
pyro.clear_param_store()
net = torch.nn.Linear(3, 1)
pyro.module("net", net)
store_names = sorted(pyro.get_param_store().keys())
print("parameter store:", store_names)
assert store_names == ["net$$$bias", "net$$$weight"], store_names

In [ ]:
# Example 5: Posterior predictive samples with Predictive and an autoguide
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoNormal
from pyro.optim import Adam

pyro.clear_param_store()
pyro.set_rng_seed(4)
x = torch.linspace(-2.0, 2.0, 50)
y = 2.0 * x + 1.0 + 0.2 * torch.randn(50)

def line_model(x, y=None):
    slope = pyro.sample("slope", dist.Normal(0.0, 5.0))
    intercept = pyro.sample("intercept", dist.Normal(0.0, 5.0))
    with pyro.plate("data", len(x)):
        return pyro.sample("obs", dist.Normal(slope * x + intercept, 0.2), obs=y)

guide = AutoNormal(line_model)
svi = SVI(line_model, guide, Adam({"lr": 0.05}), loss=Trace_ELBO())
for _ in range(600):
    svi.step(x, y)

predictive = Predictive(line_model, guide=guide, num_samples=200)
draws = predictive(x)
print({name: tuple(value.shape) for name, value in draws.items()})
assert draws["obs"].shape == (200, 50), draws["obs"].shape
assert draws["slope"].shape[0] == 200
slope_mean = draws["slope"].mean().item()
intercept_mean = draws["intercept"].mean().item()
print(f"slope {slope_mean:.2f} (true 2.0), intercept {intercept_mean:.2f} (true 1.0)")
assert abs(slope_mean - 2.0) < 0.2 and abs(intercept_mean - 1.0) < 0.2, (slope_mean, intercept_mean)